# Module 2 — Sentiment / Emotion Classifier

**Part of the RAG-Based E-commerce Customer Support Chatbot**

**Stage 2** of the pipeline: once we know the message's language, we classify its emotional
tone so the bot can react appropriately (e.g. lead with an apology if the customer is upset).

**Dataset:** [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) — English
tweets labeled with 6 emotions: `sadness, joy, love, anger, fear, surprise`.

**Mapping to 3 unified buckets** (as required by the project spec):
- `negative` ← sadness, anger, fear
- `positive` ← joy, love
- `neutral`  ← surprise

**Model:** We fine-tune `distilbert-base-uncased` (lightweight, ~66M params, fast at inference —
appropriate for a real-time chatbot) on the 3-class remapped labels, rather than reusing an
off-the-shelf sentiment pipeline, so the label space exactly matches what the Flask app expects.

**Output artifact:** a fine-tuned model + tokenizer saved to
`sentiment_model/` (loadable with `AutoModelForSequenceClassification.from_pretrained`).


## 1. Install dependencies

In [1]:
!pip install -q datasets transformers accelerate evaluate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import classification_report, accuracy_score

RANDOM_STATE = 42
MODEL_NAME = "distilbert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


## 3. Load dataset and remap 6 emotions -> 3 sentiment buckets

In [3]:
raw = load_dataset("dair-ai/emotion")
print(raw)

# dair-ai/emotion label ints (in the standard "split" config) map to:
# 0 sadness, 1 joy, 2 love, 3 anger, 4 fear, 5 surprise
id2emotion = raw["train"].features["label"].int2str

EMOTION_TO_SENTIMENT = {
    "sadness": "negative",
    "anger": "negative",
    "fear": "negative",
    "joy": "positive",
    "love": "positive",
    "surprise": "neutral",
}

SENTIMENT_LABELS = ["negative", "neutral", "positive"]  # fixed order -> ids 0,1,2
sentiment2id = {s: i for i, s in enumerate(SENTIMENT_LABELS)}

def remap_labels(example):
    emotion_str = id2emotion(example["label"])
    sentiment_str = EMOTION_TO_SENTIMENT[emotion_str]
    example["sentiment_label"] = sentiment2id[sentiment_str]
    return example

raw = raw.map(remap_labels)
print(raw["train"][0])


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'text': 'i didnt feel humiliated', 'label': 0, 'sentiment_label': 0}


In [4]:
for split in raw:
    df = raw[split].to_pandas()
    print(split, "distribution:")
    print(df["sentiment_label"].map(lambda i: SENTIMENT_LABELS[i]).value_counts())
    print()


train distribution:
sentiment_label
negative    8762
positive    6666
neutral      572
Name: count, dtype: int64

validation distribution:
sentiment_label
negative    1037
positive     882
neutral       81
Name: count, dtype: int64

test distribution:
sentiment_label
negative    1080
positive     854
neutral       66
Name: count, dtype: int64



## 4. Tokenization

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = raw.map(tokenize_fn, batched=True)
tokenized = tokenized.remove_columns(["text", "label"])
tokenized = tokenized.rename_column("sentiment_label", "labels")
tokenized.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

## 5. Load model and configure training

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(SENTIMENT_LABELS),
    id2label={i: l for i, l in enumerate(SENTIMENT_LABELS)},
    label2id=sentiment2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./sentiment_train_output",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,          # small dataset (~20k rows) + distilbert -> 2 epochs is enough
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Fine-tune

In [7]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.136339,0.090140,0.969000
2,0.044949,0.080570,0.975000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.1415240557193756, metrics={'train_runtime': 170.6545, 'train_samples_per_second': 187.513, 'train_steps_per_second': 5.86, 'total_flos': 431520246535680.0, 'train_loss': 0.1415240557193756, 'epoch': 2.0})

## 7. Evaluate on the held-out test split

In [8]:
test_predictions = trainer.predict(tokenized["test"])
test_preds = np.argmax(test_predictions.predictions, axis=-1)
test_labels = test_predictions.label_ids

test_acc = accuracy_score(test_labels, test_preds)
print(f"Test accuracy: {test_acc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=SENTIMENT_LABELS))


Test accuracy: 0.9730

              precision    recall  f1-score   support

    negative       0.97      0.98      0.98      1080
     neutral       0.79      0.70      0.74        66
    positive       0.98      0.98      0.98       854

    accuracy                           0.97      2000
   macro avg       0.92      0.89      0.90      2000
weighted avg       0.97      0.97      0.97      2000



## 8. Reusable inference function

`classify_sentiment(text)` wraps the fine-tuned model behind a simple function, matching the
interface the Flask app (Notebook 5) expects.


In [9]:
model.eval()
model.to(device)

@torch.no_grad()
def classify_sentiment(text: str) -> dict:
    """Classify the sentiment of a customer message into negative/neutral/positive.

    Args:
        text: raw customer message (any length; truncated to 128 tokens).

    Returns:
        {"sentiment": <str>, "confidence": <float 0-1>}
    """
    if not text or not text.strip():
        return {"sentiment": "neutral", "confidence": 0.0}

    inputs = tokenizer(text, truncation=True, max_length=128, return_tensors="pt").to(device)
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return {
        "sentiment": SENTIMENT_LABELS[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
    }


# Quick smoke test
for sample in [
    "This is the third time my order has been late, I am furious!",
    "Thanks so much, my package arrived perfectly and on time!",
    "Can you tell me the status of order #12345?",
]:
    print(sample, "->", classify_sentiment(sample))


This is the third time my order has been late, I am furious! -> {'sentiment': 'negative', 'confidence': 0.9963}
Thanks so much, my package arrived perfectly and on time! -> {'sentiment': 'positive', 'confidence': 0.9889}
Can you tell me the status of order #12345? -> {'sentiment': 'positive', 'confidence': 0.4941}


## 9. Mount Google Drive and set up the project folder structure

All 4 notebooks share one Drive folder, `RAG_chatbot_project/`, organized into one subfolder
per module so artifacts never collide and `app.py` can load each module from a predictable
path:

```
RAG_chatbot_project/
├── language_detection/      <- Notebook 1 saves here
├── sentiment_classifier/    <- this notebook saves here (model + tokenizer files)
├── intent_classifier/       <- Notebook 3 saves here
└── rag_pipeline/            <- Notebook 4 saves here
```

This cell mounts Drive and creates the full structure (safe to re-run).


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/RAG_chatbot_project"
LANGUAGE_DIR = os.path.join(BASE_DIR, "language_detection")
SENTIMENT_DIR = os.path.join(BASE_DIR, "sentiment_classifier")
INTENT_DIR = os.path.join(BASE_DIR, "intent_classifier")
RAG_DIR = os.path.join(BASE_DIR, "rag_pipeline")

for d in [LANGUAGE_DIR, SENTIMENT_DIR, INTENT_DIR, RAG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Project folder structure ready under:", BASE_DIR)


Mounted at /content/drive
Drive mounted. Project folder structure ready under: /content/drive/MyDrive/RAG_chatbot_project


## 10. Save artifacts

In [11]:
model.save_pretrained(SENTIMENT_DIR)
tokenizer.save_pretrained(SENTIMENT_DIR)

print("Saved sentiment model + tokenizer to:", SENTIMENT_DIR)
print(f"Final test accuracy: {test_acc:.4f}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved sentiment model + tokenizer to: /content/drive/MyDrive/RAG_chatbot_project/sentiment_classifier
Final test accuracy: 0.9730
